# Phase 7: Focus-up — blend, route, correct, assemble

Inputs: the full-sweep table (`pf_sweep_results.csv`) and the saved per-well
predictions (`pf_preds/*.npz`) from notebook 6. Reference points: floor 13.42,
best single variant 10.78 (`pf_seedens_hold0.2`), oracle 6.13 (biased), public
notebooks ~7.5, leaders ~6.0.

Broad strokes, one lever per section, every gain measured **out-of-fold**
(5-fold over wells) so we never again steer by in-sample mirages:

- **A. Prediction averaging** — uniform and fitted (NNLS) blends of the saved
  diverse predictions. Cheapest likely gain (inter-variant corr ≈ 0.69).
- **B. Observable router** — per-well variant routing. ⚠️ The bucket tables so
  far used the *true* eval span, which does **not exist at inference**. The
  router here may only use observables: `n_eval`, `z_span`, `gr_misfit`.
- **C. XHIGH mini-sweep** — specialized PF configs for the high-movement cohort
  (~⅓ of all error mass). Cohort selected by *observable* proxy.
- **D. Residual engine** — a GBM predicting `true − blend` per row from
  observable features (the "second engine" of the public recipe, built
  extrapolation-safe: trees only see a bounded residual).
- **E. Recipe assembly** — final OOF score (pooled-row RMSE, the competition's
  pooling, plus mean-per-well), recipe saved to JSON for the submission notebook.

## Setup

In [1]:
import warnings; warnings.filterwarnings("ignore")
import json, sys, time
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path("../src").resolve()))
from rogii_wellbore import clean  # noqa: E402

cfg = clean.load_config("../data/interim/clean_config.json")
CLEAN_DIR = Path("../data/interim/clean")
PRED_DIR = Path("../data/interim/pf_preds")
SWEEP_CSV = Path("../data/interim/pf_sweep_results.csv")
REAL_EVAL_FRAC = 0.73
N_FOLDS = 5

def rmse(a, b):
    return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))

def tail_mask(n, frac):
    k = int(round(n * frac)); m = np.zeros(n, bool)
    if k: m[n - k:] = True
    return m

# ---- schema discipline: explicit allow-lists (lesson from the gr_misfit leak) ----
PRED_KEYS = ["pf_spread2", "pf_seedens", "pf_base", "pf_N300", "pf_spread8", "beam_cons"]
OBS_FEATURES = ["n_eval", "n_known", "z_span", "gr_misfit"]   # observable at inference
META = ["well", "eval_span"] + OBS_FEATURES                    # eval_span = diagnosis only

sweep = pd.read_csv(SWEEP_CSV, dtype={"well": str})
ARM_COLS = [c for c in sweep.columns if c not in META]
print(f"sweep table: {len(sweep)} wells, {len(ARM_COLS)} arms")

sweep table: 773 wells, 33 arms


In [2]:
print("PRED_DIR:", PRED_DIR.resolve(), "| exists:", PRED_DIR.exists())
npz_files = sorted(PRED_DIR.glob("*.npz")) if PRED_DIR.exists() else []
print(f"{len(npz_files)} .npz files found")
if npz_files:
    z0 = np.load(npz_files[0])
    print("sample:", npz_files[0].name, "| keys:", sorted(z0.files))
    print("expected keys:", sorted(PRED_KEYS + ["true", "hold"]))
    csv_wells, file_wells = set(sweep["well"]), {p.stem for p in npz_files}
    print(f"CSV wells without a file: {len(csv_wells - file_wells)} | files not in CSV: {len(file_wells - csv_wells)}")
else:
    # hunt for where they actually got written (CWD mismatch is the usual culprit)
    hits = [str(p) for p in Path("..").rglob("*.npz")][:5]
    print("searching ../ for npz files:", hits or "none found")

PRED_DIR: C:\Users\s159286\Documents\venv\ROGII-Wellbore\data\interim\pf_preds | exists: True
773 .npz files found
sample: 000d7d20.npz | keys: ['beam_cons', 'hold', 'pf_N300', 'pf_base', 'pf_seedens', 'pf_spread2', 'pf_spread8', 'true']
expected keys: ['beam_cons', 'hold', 'pf_N300', 'pf_base', 'pf_seedens', 'pf_spread2', 'pf_spread8', 'true']
CSV wells without a file: 0 | files not in CSV: 0


In [3]:
# load saved predictions
wells, P = [], {}
skip = {"no_file": 0, "bad_load": 0, "missing_keys": 0}
key_example = None
for w in sweep["well"]:
    f = PRED_DIR / f"{w}.npz"
    if not f.exists():
        skip["no_file"] += 1; continue
    try:
        z = np.load(f)
    except Exception:
        skip["bad_load"] += 1; continue
    need = set(PRED_KEYS + ["true", "hold"])
    if not need <= set(z.files):
        skip["missing_keys"] += 1
        if key_example is None:
            key_example = (w, sorted(need - set(z.files)), sorted(z.files))
        continue
    P[w] = {k: z[k].astype(float) for k in z.files}
    wells.append(w)

print(f"loaded {len(wells)} | skipped: {skip}")
if key_example:
    print(f"e.g. well {key_example[0]} is missing {key_example[1]}; file contains {key_example[2]}")
assert wells, "No predictions loaded — fix per the skip reasons above before proceeding."

# deterministic well-level folds
rng = np.random.default_rng(0)
fold_of = dict(zip(sorted(wells), rng.permutation(len(wells)) % N_FOLDS))
sweep["fold"] = sweep["well"].map(fold_of)

def pooled_rmse(pred_by_well):
    # competition-style pooling: all rows of all wells together
    e = np.concatenate([pred_by_well[w] - P[w]["true"] for w in pred_by_well])
    return float(np.sqrt(np.mean(e ** 2)))

def per_well_mean(pred_by_well):
    return float(np.mean([rmse(pred_by_well[w], P[w]["true"]) for w in pred_by_well]))

floor_pooled = pooled_rmse({w: P[w]["hold"] for w in wells})
print(f"floor: pooled {floor_pooled:.2f} | mean-per-well "
      f"{per_well_mean({w: P[w]['hold'] for w in wells}):.2f}")

loaded 773 | skipped: {'no_file': 0, 'bad_load': 0, 'missing_keys': 0}
floor: pooled 16.37 | mean-per-well 13.42


## A. Prediction averaging (uniform + fitted NNLS), out-of-fold

Components: the six saved predictions + `hold`. Uniform average first; then
non-negative least squares weights fitted on the training folds' pooled rows and
applied to held-out wells. NNLS keeps weights interpretable (a true blend).

In [4]:
from scipy.optimize import nnls

COMPS = PRED_KEYS + ["hold"]

def stack(wlist, max_rows=300_000, seed=0):
    Xs, ys = [], []
    for w in wlist:
        cols = np.column_stack([P[w][k] for k in COMPS])
        Xs.append(cols); ys.append(P[w]["true"])
    X = np.vstack(Xs); y = np.concatenate(ys)
    if len(y) > max_rows:
        idx = np.random.default_rng(seed).choice(len(y), max_rows, replace=False)
        X, y = X[idx], y[idx]
    return X, y

uni = {w: np.mean([P[w][k] for k in PRED_KEYS], axis=0) for w in wells}
print(f"uniform mean of {len(PRED_KEYS)} preds: pooled {pooled_rmse(uni):.2f} | "
      f"per-well {per_well_mean(uni):.2f}")

oof_nnls, fold_w = {}, {}
for f in range(N_FOLDS):
    tr = [w for w in wells if fold_of[w] != f]
    va = [w for w in wells if fold_of[w] == f]
    X, y = stack(tr)
    wts, _ = nnls(X, y)
    s = wts.sum(); wts = wts / s if s > 0 else np.ones(len(COMPS)) / len(COMPS)
    fold_w[f] = wts
    for w in va:
        oof_nnls[w] = np.column_stack([P[w][k] for k in COMPS]) @ wts
print(f"NNLS blend (OOF):            pooled {pooled_rmse(oof_nnls):.2f} | "
      f"per-well {per_well_mean(oof_nnls):.2f}")
wbar = np.mean([fold_w[f] for f in range(N_FOLDS)], axis=0)
print("mean NNLS weights:", {k: round(float(v), 3) for k, v in zip(COMPS, wbar) if v > 0.01})

uniform mean of 6 preds: pooled 13.09 | per-well 10.68
NNLS blend (OOF):            pooled 13.12 | per-well 10.65
mean NNLS weights: {'pf_spread2': 0.195, 'pf_seedens': 0.353, 'pf_base': 0.068, 'pf_N300': 0.114, 'pf_spread8': 0.097, 'hold': 0.172}


## B. Observable-feature router (OOF)

Per training fold: find each well's best component, fit a depth-2 decision tree
on `n_eval, z_span, gr_misfit` to predict it, route held-out wells. This is the
honest version of the "forced selector" — and the comparison against Section A
tells us whether routing beats blending on this variant pool.

In [5]:
from sklearn.tree import DecisionTreeClassifier

ROUTE_OPTS = PRED_KEYS + ["hold"]
per_well_arm_rmse = {w: {k: rmse(P[w][k], P[w]["true"]) for k in ROUTE_OPTS} for w in wells}

oof_route, route_pick = {}, {}
F = sweep.set_index("well")[OBS_FEATURES]
for f in range(N_FOLDS):
    tr = [w for w in wells if fold_of[w] != f]
    va = [w for w in wells if fold_of[w] == f]
    ybest = [min(per_well_arm_rmse[w], key=per_well_arm_rmse[w].get) for w in tr]
    clf = DecisionTreeClassifier(max_depth=2, min_samples_leaf=max(3, len(tr) // 10),
                                 random_state=0).fit(F.loc[tr].values, ybest)
    for w in va:
        pick = clf.predict(F.loc[[w]].values)[0]
        route_pick[w] = pick
        oof_route[w] = P[w][pick]
print(f"router (OOF):   pooled {pooled_rmse(oof_route):.2f} | per-well {per_well_mean(oof_route):.2f}")
print("route distribution:", pd.Series(route_pick).value_counts().to_dict())
print("\n(if the router loses to the NNLS blend, blending wins on this pool — "
      "consistent with winner-spread being largely noise)")

router (OOF):   pooled 14.02 | per-well 11.53
route distribution: {np.str_('pf_seedens'): 316, np.str_('pf_N300'): 223, np.str_('pf_spread2'): 104, np.str_('hold'): 88, np.str_('pf_spread8'): 42}

(if the router loses to the NNLS blend, blending wins on this pool — consistent with winner-spread being largely noise)


## C. XHIGH mini-sweep — specialized configs for the high-movement cohort

XHIGH (~18% of wells, ~⅓ of error mass) is where every config struggles. Cohort
selection must be **observable**: we use a `z_span` threshold (calibrated below
against the true-span diagnosis). Specialized configs: wider spread, hotter
rate noise, longer windows. Re-runs the PF, so this is the slow cell
(~cohort × 6 configs × ~0.5 s).

In [6]:
# how well does observable z_span identify true-XHIGH wells? (diagnosis only)
xh_true = sweep["eval_span"] > 40
if xh_true.any():
    from sklearn.metrics import roc_auc_score
    try:
        auc = roc_auc_score(xh_true, sweep["z_span"])
        print(f"z_span identifies true XHIGH: AUC {auc:.2f}")
    except Exception:
        pass
ZTHR = float(sweep.loc[xh_true, "z_span"].quantile(0.25)) if xh_true.any() \
       else float(sweep["z_span"].quantile(0.8))
cohort = sweep.loc[sweep["z_span"] >= ZTHR, "well"].tolist()
print(f"observable cohort (z_span >= {ZTHR:.0f}): {len(cohort)} wells "
      f"(true-XHIGH capture: {sweep.loc[sweep['well'].isin(cohort),'eval_span'].gt(40).mean():.0%})")

z_span identifies true XHIGH: AUC 0.51
observable cohort (z_span >= 679): 593 wells (true-XHIGH capture: 17%)


In [7]:
def _prep_well(hz, tw, frac):
    tw_s = tw.sort_values("TVT")
    return dict(
        twt=tw_s["TVT"].values.astype(float),
        twg=tw_s["GR"].ffill().bfill().values.astype(float),
        tvt=hz["TVT"].values.astype(float),
        Z=hz["Z"].values.astype(float), MD=hz["MD"].values.astype(float),
        gr=pd.Series(hz["GR"].values).interpolate(limit_direction="both")
            .fillna(90.0).values)

def run_pf(Pw, N=500, spread=4.5, MOM=0.998, VN=0.002, PN=0.005,
           rate_win=30, gs_max=60.0, seed=42):
    twt, twg, tvt, Z, MD, gr = (Pw[k] for k in ("twt", "twg", "tvt", "Z", "MD", "gr"))
    kn, ev = Pw["kn"], Pw["ev"]; last = kn[-1]
    gs = float(np.clip(np.nanstd(gr[kn] - np.interp(tvt[kn], twt, twg)), 10.0, gs_max))
    tl = kn[-rate_win:]
    dt = np.diff(tvt[tl]); dz = np.diff(Z[tl]); dm = np.diff(MD[tl]); ok = dm > 0
    ir = float(np.median((dt + dz)[ok] / dm[ok])) if ok.sum() >= 3 else 0.0
    rng = np.random.default_rng(seed)
    pos = (tvt[last] + Z[last]) + spread * rng.standard_normal(N)
    rate = ir + 0.01 * rng.standard_normal(N)
    w = np.ones(N) / N; out = np.empty(len(ev)); prev = MD[last]
    lo, hi = twt[0] - 100, twt[-1] + 100
    for i, idx in enumerate(ev):
        dmS = max(MD[idx] - prev, 1.0)
        rate = MOM * rate + VN * rng.standard_normal(N)
        pos = pos + rate * dmS + PN * rng.standard_normal(N)
        tvt_p = np.clip(pos - Z[idx], lo, hi); pos = tvt_p + Z[idx]
        g = gr[idx]
        if np.isfinite(g):
            d = (g - np.interp(tvt_p, twt, twg)) / gs
            w = w * np.maximum(np.exp(-0.5 * np.minimum(d * d, 600.0)), 1e-300)
            s = w.sum(); w = w / s if s > 0 else np.ones(N) / N
        if 1.0 / np.sum(w * w) < 0.5 * N:
            ci = np.clip(np.searchsorted(np.cumsum(w),
                 (np.arange(N) + rng.uniform(0, 1)) / N), 0, N - 1)
            pos = pos[ci] + 0.1 * rng.standard_normal(N)
            rate = rate[ci] + 0.001 * rng.standard_normal(N)
            w = np.ones(N) / N
        out[i] = np.sum(w * (pos - Z[idx])); prev = MD[idx]
    return out

XH_CONFIGS = {
    "xh_spread8_vn01":  dict(spread=8.0,  VN=0.01),
    "xh_spread12_vn01": dict(spread=12.0, VN=0.01),
    "xh_spread8_rw90":  dict(spread=8.0,  rate_win=90),
    "xh_spread12_N1k":  dict(spread=12.0, N=1000),
    "xh_vn02":          dict(VN=0.02),
    "xh_spread8_mom99": dict(spread=8.0,  MOM=0.99),
}

xh_rows = []
t0 = time.time()
for w_i, wid in enumerate(cohort):
    hz = pd.read_csv(CLEAN_DIR / "train" / f"{wid}__horizontal_well.csv",
                     dtype={"well_id": str}).sort_values("MD").reset_index(drop=True)
    tw = pd.read_csv(CLEAN_DIR / "train" / f"{wid}__typewell.csv", dtype={"well_id": str})
    Pw = _prep_well(hz, tw, REAL_EVAL_FRAC)
    m = tail_mask(len(hz), REAL_EVAL_FRAC)
    Pw["kn"] = np.where(~m)[0]; Pw["ev"] = np.where(m)[0]
    true = Pw["tvt"][Pw["ev"]]
    rec = {"well": wid, "base_blend": rmse(np.mean([P[wid][k] for k in PRED_KEYS], 0), true)
           if wid in P else np.nan}
    for name, kw in XH_CONFIGS.items():
        rec[name] = rmse(run_pf(Pw, **kw), true)
    xh_rows.append(rec)
    if (w_i + 1) % 25 == 0:
        print(f"  ...{w_i+1}/{len(cohort)}  [{(time.time()-t0)/60:.1f} min]")
xh = pd.DataFrame(xh_rows)
print("\nXHIGH cohort mean RMSE:")
print(xh.drop(columns=["well"]).mean().sort_values().round(2))

  ...25/593  [0.5 min]
  ...50/593  [1.2 min]
  ...75/593  [2.0 min]
  ...100/593  [3.0 min]
  ...125/593  [4.1 min]
  ...150/593  [4.8 min]
  ...175/593  [5.3 min]
  ...200/593  [5.8 min]
  ...225/593  [6.3 min]
  ...250/593  [6.8 min]
  ...275/593  [7.3 min]
  ...300/593  [7.7 min]
  ...325/593  [8.2 min]
  ...350/593  [8.7 min]
  ...375/593  [9.2 min]
  ...400/593  [9.6 min]
  ...425/593  [10.1 min]
  ...450/593  [10.6 min]
  ...475/593  [11.1 min]
  ...500/593  [11.6 min]
  ...525/593  [12.0 min]
  ...550/593  [12.5 min]
  ...575/593  [13.0 min]

XHIGH cohort mean RMSE:
base_blend          10.59
xh_spread8_rw90     13.12
xh_spread12_N1k     14.57
xh_vn02             14.91
xh_spread8_vn01     15.24
xh_spread12_vn01    15.95
xh_spread8_mom99    17.37
dtype: float64


## D. Residual engine (the GBM second engine, extrapolation-safe)

Per eval row, predict `true − blend` from observable features only; trees see a
bounded residual so the extrapolation trap can't bite. OOF by the same well
folds; final = `blend + α·residual`, α fitted per fold on the training wells.

In [8]:
try:
    import lightgbm as lgb
    def make_gbm(): return lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05,
                                             num_leaves=63, verbose=-1, random_state=0)
except Exception:
    from sklearn.ensemble import HistGradientBoostingRegressor
    def make_gbm(): return HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05,
                                                         random_state=0)

# choose the base blend = OOF NNLS (Section A); fall back to uniform if missing
BASE_BLEND = {w: oof_nnls.get(w, uni[w]) for w in wells}

def row_feats(wid):
    hz = pd.read_csv(CLEAN_DIR / "train" / f"{wid}__horizontal_well.csv",
                     dtype={"well_id": str}).sort_values("MD").reset_index(drop=True)
    tw = pd.read_csv(CLEAN_DIR / "train" / f"{wid}__typewell.csv", dtype={"well_id": str})
    Pw = _prep_well(hz, tw, REAL_EVAL_FRAC)
    m = tail_mask(len(hz), REAL_EVAL_FRAC)
    kn = np.where(~m)[0]; ev = np.where(m)[0]
    Z, MD, gr = Pw["Z"], Pw["MD"], Pw["gr"]
    twt, twg = Pw["twt"], Pw["twg"]
    blend = BASE_BLEND[wid]
    depth = (MD[ev] - MD[kn[-1]])                       # distance into eval
    dz = np.gradient(Z)[ev]
    gmis = gr[ev] - np.interp(blend, twt, twg)          # GR misfit at predicted TVT
    spread = np.std(np.column_stack([P[wid][k] for k in PRED_KEYS]), axis=1)  # ens spread
    X = np.column_stack([depth, dz, gr[ev], gmis, spread,
                         np.full(len(ev), float(Z.max() - Z.min()))])
    y = Pw["tvt"][ev] - blend
    return X, y

feats, targs = {}, {}
for wid in wells:
    try:
        feats[wid], targs[wid] = row_feats(wid)
    except Exception:
        continue
usable = sorted(feats)
print(f"row features built for {len(usable)} wells")

oof_resid = {}
ALPHAS = (0.0, 0.3, 0.5, 0.7, 1.0)
for f in range(N_FOLDS):
    tr = [w for w in usable if fold_of[w] != f]
    va = [w for w in usable if fold_of[w] == f]
    if not tr or not va: continue
    Xtr = np.vstack([feats[w] for w in tr]); ytr = np.concatenate([targs[w] for w in tr])
    mdl = make_gbm().fit(Xtr, ytr)
    # fit alpha on the TRAINING wells (pooled), apply to validation
    tr_pred = {w: mdl.predict(feats[w]) for w in tr}
    best_a, best_r = 0.0, np.inf
    for a in ALPHAS:
        e = np.concatenate([BASE_BLEND[w] + a * tr_pred[w] - P[w]["true"] for w in tr])
        r = float(np.sqrt(np.mean(e ** 2)))
        if r < best_r: best_a, best_r = a, r
    for w in va:
        oof_resid[w] = BASE_BLEND[w] + best_a * mdl.predict(feats[w])
print(f"residual engine (OOF): pooled {pooled_rmse(oof_resid):.2f} | "
      f"per-well {per_well_mean(oof_resid):.2f}")

row features built for 773 wells
residual engine (OOF): pooled 14.28 | per-well 11.80


## E. Assemble the recipe + final OOF table

Compare every stage on both metrics; save the winning recipe and the OOF
predictions. The recipe JSON is what the submission notebook (08) consumes.

In [9]:
candidates = {
    "floor":            {w: P[w]["hold"] for w in wells},
    "best_single_seh2": {w: 0.8 * P[w]["pf_seedens"] + 0.2 * P[w]["hold"] for w in wells},
    "uniform_blend":    uni,
    "nnls_blend_oof":   oof_nnls,
    "router_oof":       oof_route,
    "resid_engine_oof": {w: oof_resid.get(w, BASE_BLEND[w]) for w in wells},
}
print(f"{'stage':22s} {'pooled':>8s} {'per-well':>9s}")
rows = []
for k, pv in candidates.items():
    pr, pw = pooled_rmse(pv), per_well_mean(pv)
    rows.append((k, pr, pw))
    print(f"{k:22s} {pr:8.3f} {pw:9.3f}")
winner = min(rows[1:], key=lambda r: r[1])   # exclude floor from winning
print(f"\nWINNER: {winner[0]}  (pooled {winner[1]:.3f})")

RECIPE = {
    "pred_components": PRED_KEYS,
    "nnls_mean_weights": {k: float(v) for k, v in zip(COMPS, wbar)},
    "winner_stage": winner[0],
    "xh_cohort_rule": {"feature": "z_span", "threshold": ZTHR},
    "xh_best_config": xh.drop(columns=["well"]).mean().drop("base_blend").idxmin()
                      if len(xh) else None,
    "residual_alphas_tried": list(ALPHAS),
    "n_folds": N_FOLDS,
    "mask_frac_assumed": REAL_EVAL_FRAC,
}
with open("../data/interim/recipe_v1.json", "w") as f:
    json.dump(RECIPE, f, indent=2, default=str)
np.savez_compressed("../data/interim/oof_final.npz",
                    **{w: candidates[winner[0]][w].astype(np.float32) for w in wells})
print("\nsaved recipe_v1.json + oof_final.npz")

stage                    pooled  per-well
floor                    16.371    13.423
best_single_seh2         13.255    10.776
uniform_blend            13.088    10.681
nnls_blend_oof           13.119    10.646
router_oof               14.022    11.526
resid_engine_oof         14.284    11.802

WINNER: uniform_blend  (pooled 13.088)

saved recipe_v1.json + oof_final.npz


## Iteration knobs (in expected-value order)

1. **Mask realism**: rerun the winner at per-well masks drawn from the test
   distribution (0.67–0.80) instead of flat 0.73 before trusting CV→LB.
2. **XHIGH routing**: if a `xh_*` config beat `base_blend` on the cohort, add it
   as a routed component (observable rule already in the recipe) and re-run E.
3. **Richer residual features**: typewell local slope at predicted TVT,
   per-variant disagreement directions, formation (`Geology_canon`) one-hots.
4. **More saved components**: persist beam variants + hold-blends as predictions
   in notebook 6 and let NNLS see them all.
5. **Seed-ensemble the NNLS winner components** (multi-seed each PF entry).
6. Then notebook 8: inference on `data/raw/test` with this recipe → submission.